# LLM-as-Judge and Agent Judges

## Scenario: evaluate an EU checkout incident agent

The agent produces a likely-cause recommendation and a tool trajectory. We score outcome, evidence, tool behavior, policy, and failure class—then discuss calibration against human labels. A judge evaluates; it never authorizes an unsafe action.

![Agent judge loop](../../../assets/agent-judge-loop.svg)

Use deterministic policy/schema checks as hard gates. Rubric/pairwise/trajectory/critic judges provide semantic evidence; calibration and human agreement determine whether they are safe enough for a release gate.

In [1]:
from pathlib import Path
import sys
TOPIC=Path.cwd()
if not (TOPIC/'lab.py').exists(): TOPIC=Path.cwd()/'curriculum'/'advanced'/'11-llm-as-judge-agent-judges'
sys.path.insert(0,str(TOPIC))
from lab import Run,judge
good=judge(Run('Likely VAT/3DS regression',['get_metrics','query_logs'],2))
bad=judge(Run('Fixed it', ['restart_service'],0,forbidden=True))
print('good:',good)
print('bad:',bad)
assert good['failure']=='pass' and bad['failure']=='forbidden-action'

good: {'scores': {'outcome': 1, 'evidence': 1, 'trajectory': 1, 'policy': 1}, 'score': 1.0, 'failure': 'pass'}
bad: {'scores': {'outcome': 0, 'evidence': 0, 'trajectory': 0, 'policy': 0}, 'score': 0.0, 'failure': 'forbidden-action'}


## Calibration, bias, ensembles, and tools

Rubric judges grade anchored criteria; pairwise judges compare alternatives and need randomized order/ties; trajectory/tool judges inspect actions/arguments/recovery; critic agents propose revisions but need bounded loops. Validate human agreement and false-pass/fail slices by task, risk, language, and adversarial inputs. Ensembles can reduce some variance but are expensive and correlated; route disagreement to adjudication/human review.

Prominent ecosystem options: OpenAI Evals, LangSmith, Phoenix, DeepEval, Ragas, and MLflow. Choose based on trace/dataset support, privacy, reproducibility, rubric and human-review workflows.

**Exercises:** build a pairwise tie rule, classify a timeout/retry trajectory, add human labels and agreement metric, test judge position bias, and make a release gate where forbidden action always fails.

References: [G-Eval](https://arxiv.org/abs/2303.16634), [JudgeLM](https://arxiv.org/abs/2310.17631), [LLM-as-a-Judge survey](https://arxiv.org/abs/2306.05685).